# 🔎 مشروع RAG للإجابة الذكية على أسئلة جامعة الملك خالد

هذا النوتبوك يوضح خطوة بخطوة كيف يمكنك تحويل ملف Markdown منسق إلى نظام استرجاع وتوليد إجابات ذكي (RAG) مع دعم السياق ودمج العناوين، باستخدام مكتبة Sentence Transformers وواجهة LLM للتوليد النهائي.


## 1️⃣ الإعدادات وتحميل الحزم

تأكد من تثبيت جميع الحزم المطلوبة:

```python
!pip install transformers sentence-transformers scikit-learn numpy



---
# 1️⃣ الإعدادات العامة


In [2]:
from utils.chunking import MDChunker
from utils.embedding import Embedder
from utils.retrieval import Retriever
from utils.generation import build_prompt

import os

## 2️⃣ تقسيم ملف Markdown إلى "Chunks" جاهزة للـ Embedding

سنقوم بتحميل الملف وتقطيعه حسب العناوين والفقرات مع دعم تداخل (overlap) للحفاظ على السياق.


In [4]:
# إعداد chunker
chunker = MDChunker(max_tokens=380, overlap=30)

# استخراج جميع الـchunks من الملف
kk_md_file = "data/kk_data.md"    # عدّل المسار حسب مشروعك
all_chunks = chunker.process_md_file(kk_md_file)

print(f"عدد الـchunks المستخرجة: {len(all_chunks)}")
print("أول مثال:")
print(all_chunks[0])


Token indices sequence length is longer than the specified maximum sequence length for this model (929 > 512). Running this sequence through the model will result in indexing errors


عدد الـchunks المستخرجة: 112
أول مثال:
{'chunk_text': 'انا عبارة عن نموذج راج تتمثل مهمتي في تقديم معلومات مفصلة وشاملة حول جامعة الملك خالد، موجهة للطلاب وأعضاء هيئة التدريس والإداريين، مع التركيز على البيانات الحديثة للعام الحالي حيثما أمكن. تم تطويري كجزء من مشروع تخرج بواسطة مجموعة من الطلاب في جامعة الملك خالد في المملكة العربية السعودية تحت اشراف الدكتور (Dr. Ayman Alhalaybeh) وتنفيذ الطلاب: Anwr Ebrahim Ahmad Zayed\t\nMarwan Sultan Alamari\t\nAhmed Ibraheem Alnami\t\nMohamed Saad Aljohany', 'section_title': 'من انت', 'section_idx': 0, 'chunk_id': 0, 'token_start': 0, 'token_end': 305}


## 3️⃣ استخراج الـ Embeddings وتخزين النتائج

سنحوّل كل chunk إلى تمثيل رقمي (embedding) باستخدام sentence-transformers/all-mpnet-base-v2 ونحفظها مع metadata.


In [5]:
embedder = Embedder(model_name="all-mpnet-base-v2")

# حساب الـembeddings
embeddings = embedder.encode_chunks(all_chunks)

# حفظ النتائج (تلقائياً في مجلد embeddings/)
embedder.save_embeddings(embeddings, all_chunks)
print("تم حفظ النتائج!")

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

تم حفظ النتائج!


## 4️⃣ تحميل النتائج واختبار دالة الاسترجاع

سنحمل المصفوفة المحفوظة ونبني دالة الاسترجاع مع دعم السياق الجانبي.


In [6]:
# تحميل embeddings وmetadata
embeddings, metadata = embedder.load_embeddings()

# تجهيز نموذج sentence-transformers (مطلوب للاستعلام)
from sentence_transformers import SentenceTransformer
retrieval_model = SentenceTransformer('all-mpnet-base-v2')

# تهيئة الريتريفر
retriever = Retriever(embeddings, metadata, retrieval_model)

# جرب سؤالاً
query = "ماهي شروط القبول في كلية الطب؟"
results = retriever.retrieve(query, top_k=3, with_context_window=1)

for idx, r in enumerate(results):
    print(f"--- [{idx+1}] ---")
    print(f"[{r['section_title']}]")
    print(r['chunk_text'])
    print()


--- [1] ---
[الخدمات الصحية:]
توفر جامعة الملك خالد خدمات صحية للطلاب من خلال كلياتها الصحية والمدينة الطبية التابعة لها. تشمل هذه الخدمات : - * * القافلة الطبية : * * تنظم الجامعة قوافل طبية بالتعاون مع الكليات الصحية لتقديم الاستشارات الطبية في مختلف المحافظات. - * * الخدمات الطبية الطارية : * * توفر الجامعة خدمات طبية طارية للطلاب. - * * المستشفيات الجامعية : * * يستفيد الطلاب من الخدمات الصحية المقدمة في مستشفى الملك خالد الجامعي التابع لجامعة الملك سعود ، والذي يقدم خدمات الرعاية الصح

--- [2] ---
[الخدمات الصحية:]
سعود ، والذي يقدم خدمات الرعاية الصحية الاولية والثانوية والثالثية.

--- [3] ---
[الإرشاد الأكاديمي:]
يعد الارشاد الاكاديمي جزءا اساسيا من العملية التعليمية في جامعة الملك خالد ، ويهدف الى دعم الطلاب اكاديميا ونفسيا ومهنيا. يتم تقديم خدمات الارشاد من خلال : - * * وحدات الارشاد الاكاديمي في الكليات : * * كل كلية لديها وحدة ارشاد اكاديمي خاصة بها ، تتكون من اعضاء هيية تدريس يقدمون الاستشارات والتوجيه للطلاب. - * * مركز التوجيه والارشاد بعمادة شوون الطلاب : * * يقدم استشار

## 5️⃣ بناء البرومبت لواجهة LLM

نقوم بدمج النتائج مع عناوينها لبناء نص سياقي قوي لنموذج التوليد.


In [7]:
prompt = build_prompt(query, results)
print(prompt)

استخدم المعلومات التالية للإجابة عن السؤال بدقة، مع ذكر التفاصيل عند توفرها.

المعلومات:
[الخدمات الصحية:]
توفر جامعة الملك خالد خدمات صحية للطلاب من خلال كلياتها الصحية والمدينة الطبية التابعة لها. تشمل هذه الخدمات : - * * القافلة الطبية : * * تنظم الجامعة قوافل طبية بالتعاون مع الكليات الصحية لتقديم الاستشارات الطبية في مختلف المحافظات. - * * الخدمات الطبية الطارية : * * توفر الجامعة خدمات طبية طارية للطلاب. - * * المستشفيات الجامعية : * * يستفيد الطلاب من الخدمات الصحية المقدمة في مستشفى الملك خالد الجامعي التابع لجامعة الملك سعود ، والذي يقدم خدمات الرعاية الصح


[الخدمات الصحية:]
سعود ، والذي يقدم خدمات الرعاية الصحية الاولية والثانوية والثالثية.


[الإرشاد الأكاديمي:]
يعد الارشاد الاكاديمي جزءا اساسيا من العملية التعليمية في جامعة الملك خالد ، ويهدف الى دعم الطلاب اكاديميا ونفسيا ومهنيا. يتم تقديم خدمات الارشاد من خلال : - * * وحدات الارشاد الاكاديمي في الكليات : * * كل كلية لديها وحدة ارشاد اكاديمي خاصة بها ، تتكون من اعضاء هيية تدريس يقدمون الاستشارات والتوجيه للطلاب. - * * مرك

## 6️⃣ توليد الإجابة النهائية باستخدام Gemma-2B-IT على جهازك

سنستخدم نموذج Gemma-2B-IT المحلي لتوليد الإجابات مباشرة بناءً على البرومبت المُجهز.


In [13]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import is_flash_attn_2_available

# حدد اسم النموذج حسب الموجود عندك
model_id = "google/gemma-2-2b-it"  # عدّل المسار إذا كنت حافظه محلياً

# إعداد quantization (اختياري لتقليل الذاكرة، مفيد إذا كانت VRAM أقل من 16G)
from transformers import BitsAndBytesConfig
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)
use_quantization_config = True

# تفعيل Flash Attention 2 إذا متاح (RTX 40xx فقط)
if (is_flash_attn_2_available()) and (torch.cuda.get_device_capability(0)[0] >= 8):
    attn_implementation = "flash_attention_2"
else:
    attn_implementation = "sdpa"
print(f"[INFO] Using attention implementation: {attn_implementation}")

# تحميل التوكنيزر
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_id)

# تحميل النموذج
llm_model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=model_id,
    torch_dtype=torch.float16,
    quantization_config=quantization_config if use_quantization_config else None,
    low_cpu_mem_usage=False,
    attn_implementation=attn_implementation
)

if not use_quantization_config:
    llm_model.to("cuda")


[INFO] Using attention implementation: sdpa


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [17]:
# def gemma_generate(prompt, max_new_tokens=256):
#     # إعداد المحادثة كما في النماذج التعليماتية
#     dialogue_template = [
#         {"role": "user", "content": prompt}
#     ]
#     formatted_prompt = tokenizer.apply_chat_template(
#         conversation=dialogue_template,
#         tokenize=False,
#         add_generation_prompt=True
#     )
#
#     # تحويل البرومبت إلى input_ids وإرساله للـGPU
#     input_ids = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")
#     # توليد الإجابة
#     with torch.no_grad():
#         outputs = llm_model.generate(
#             **input_ids,
#             max_new_tokens=max_new_tokens,
#             pad_token_id=tokenizer.eos_token_id,
#         )
#     # استخراج النص الناتج
#     outputs_decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
#     # حذف البرومبت المكرر (لو حدث)
#     answer = outputs_decoded.replace(formatted_prompt, '').replace('<bos>', '').replace('<eos>', '').strip()
#     return answer


def build_prompt_gemma(query, retrieved_chunks, max_tokens_context=1200):
    """
    بناء برومبت مناسب لنماذج Gemma لتفادي تكرار البرومبت أو طوله الزائد.
    """
    context = []
    seen = set()
    total_length = 0

    for chunk in retrieved_chunks:
        t = chunk["chunk_text"].strip()
        # تجنب تكرار النصوص
        if t in seen or not t: continue
        tokens = len(t.split())
        if total_length + tokens > max_tokens_context: break
        context.append(t)
        seen.add(t)
        total_length += tokens

    context_str = "\n".join(context)
    prompt = (
        f"لديك المعلومات التالية:\n{context_str}\n\n"
        f"باستخدام هذه المعلومات فقط، أجب بشكل منظم على السؤال التالي، مع تنسيق النقاط:\n"
        f"{query}\n\n"
        f"الجواب المنظم:"
    )
    return prompt


def gemma_generate(prompt, max_new_tokens=256):
    dialogue_template = [{"role": "user", "content": prompt}]
    formatted_prompt = tokenizer.apply_chat_template(
        conversation=dialogue_template,
        tokenize=False,
        add_generation_prompt=True
    )
    input_ids = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = llm_model.generate(
            **input_ids,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
        )
    outputs_decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # قص كل ما قبل عبارة "الجواب المنظم:"
    answer = outputs_decoded.split("الجواب المنظم:")[-1].strip()
    return answer


## 7️⃣ استعلام تفاعلي كامل (RAG: استرجاع + توليد Gemma)

اكتب سؤالك، وسيتم:
1. استرجاع أهم الـchunks مع السياق
2. تجهيز البرومبت
3. توليد الإجابة تلقائيًا من Gemma


In [18]:
def rag_answer(user_query, retriever, build_prompt_fn, llm_fn, top_k=5, context_win=1, max_new_tokens=256):
    # استرجاع السياق
    retrieved = retriever.retrieve(user_query, top_k=top_k, with_context_window=context_win)
    # بناء البرومبت
    prompt = build_prompt_fn(user_query, retrieved)
    print("----- البرومبت النهائي -----\n")
    print(prompt)
    # توليد الإجابة
    print("\n----- إجابة النموذج -----\n")
    answer = llm_fn(prompt, max_new_tokens=max_new_tokens)
    print(answer)
    return answer

# تجربة مباشرة
user_q = "ماهي أنواع المنح الدراسية المتوفرة للطلاب غير السعوديين؟"
rag_answer(user_q, retriever, build_prompt, gemma_generate)


----- البرومبت النهائي -----

استخدم المعلومات التالية للإجابة عن السؤال بدقة، مع ذكر التفاصيل عند توفرها.

المعلومات:
[المنح الدراسية:]
تقدم جامعة الملك خالد منحا دراسية للطلاب السعوديين وغير السعوديين ، بهدف دعمهم ماليا وتشجيعهم على التفوق الاكاديمي. تشمل هذه المنح : - * * منح البكالوريوس والدراسات العليا : * * تغطي هذه المنح الرسوم الدراسية بالكامل ، وتوفر سكنا مجانيا ، وتامينا صحيا ، ومكافات مالية شهرية. - * * شروط التقديم : * * تختلف شروط التقديم للمنح حسب نوع المنحة والمرحلة الدراسية ، ولكنها تتضمن عادة الحصول على معدل تراكمي معين في الثانوية العامة او البكالوريوس ، واجادة اللغة العربية لبع


[المنح الدراسية:]
##كالوريوس ، واجادة اللغة العربية لبعض التخصصات. - * * الية التقديم : * * يتم التقديم للمنح الدراسية الكترونيا عبر بوابة القبول المخصصة للمنح في موقع الجامعة ، او من خلال الادارة العامة لطلاب المنح. يتم الاعلان عن مواعيد فتح باب القبول للمنح بشكل دوري.


[أسئلة القبول والتسجيل:]
* * ما هو رقم الاتصال الخاص بعمادة القبول والتسجيل [UNK] * * يمكنك التواصل عن طريق الرقم الموحد 

'user\nاستخدم المعلومات التالية للإجابة عن السؤال بدقة، مع ذكر التفاصيل عند توفرها.\n\nالمعلومات:\n[المنح الدراسية:]\nتقدم جامعة الملك خالد منحا دراسية للطلاب السعوديين وغير السعوديين ، بهدف دعمهم ماليا وتشجيعهم على التفوق الاكاديمي. تشمل هذه المنح : - * * منح البكالوريوس والدراسات العليا : * * تغطي هذه المنح الرسوم الدراسية بالكامل ، وتوفر سكنا مجانيا ، وتامينا صحيا ، ومكافات مالية شهرية. - * * شروط التقديم : * * تختلف شروط التقديم للمنح حسب نوع المنحة والمرحلة الدراسية ، ولكنها تتضمن عادة الحصول على معدل تراكمي معين في الثانوية العامة او البكالوريوس ، واجادة اللغة العربية لبع\n\n\n[المنح الدراسية:]\n##كالوريوس ، واجادة اللغة العربية لبعض التخصصات. - * * الية التقديم : * * يتم التقديم للمنح الدراسية الكترونيا عبر بوابة القبول المخصصة للمنح في موقع الجامعة ، او من خلال الادارة العامة لطلاب المنح. يتم الاعلان عن مواعيد فتح باب القبول للمنح بشكل دوري.\n\n\n[أسئلة القبول والتسجيل:]\n* * ما هو رقم الاتصال الخاص بعمادة القبول والتسجيل [UNK] * * يمكنك التواصل عن طريق الرقم الموحد 0172418000 

In [19]:
def rag_answer(user_query, retriever, build_prompt_fn, llm_fn, top_k=3, context_win=0, max_new_tokens=256):
    retrieved = retriever.retrieve(user_query, top_k=top_k, with_context_window=context_win)
    prompt = build_prompt_fn(user_query, retrieved)
    print("----- البرومبت النهائي -----\n")
    print(prompt)
    print("\n----- إجابة النموذج -----\n")
    answer = llm_fn(prompt, max_new_tokens=max_new_tokens)
    print(answer)
    return answer

# جرب بهذا الشكل
rag_answer(
    "ماهي أنواع المنح الدراسية المتوفرة للطلاب غير السعوديين؟",
    retriever,
    build_prompt_gemma,    # استخدم البرومبت الجديد
    gemma_generate,
    top_k=2,               # قلل عدد القطع للسياق
    context_win=0,         # لا تكرر القطع المجاورة
    max_new_tokens=256
)

----- البرومبت النهائي -----

لديك المعلومات التالية:
- **منح داخلية:** للطلاب المقيمين في المملكة العربية السعودية من غير السعوديين.
- **منح خارجية:** للطلاب من خارج المملكة العربية السعودية.
على الخدمات الاساسية للطلاب من المطابخ والمغاسل وصالات الاستراحة والترفيه ، واماكن الضيافة والقراءة والمستودعات.

باستخدام هذه المعلومات فقط، أجب بشكل منظم على السؤال التالي، مع تنسيق النقاط:
ماهي أنواع المنح الدراسية المتوفرة للطلاب غير السعوديين؟

الجواب المنظم:

----- إجابة النموذج -----

model
يوجد منح دراسي متعدد لطلاب غير سعوديين،  يعتمد نوع المنح على نوعية الطالب. 

**منح داخلية:**

* تُقدم منحًا داخلية لطلاب غير سعوديين المقيمين في المملكة العربية السعودية.

**منح خارجية:**

* تُقدم منحًا خارجية لطلاب من خارج المملكة العربية السعودية.


'model\nيوجد منح دراسي متعدد لطلاب غير سعوديين،  يعتمد نوع المنح على نوعية الطالب. \n\n**منح داخلية:**\n\n* تُقدم منحًا داخلية لطلاب غير سعوديين المقيمين في المملكة العربية السعودية.\n\n**منح خارجية:**\n\n* تُقدم منحًا خارجية لطلاب من خارج المملكة العربية السعودية.'